# Substation Load Forecasting with ukpyn

This notebook demonstrates a comprehensive workflow for substation load forecasting using UK Power Networks open data. We cover two main themes:

## Theme 1: Raw Time Series Forecaster
End-to-end workflow from data acquisition to trained ML model.

## Theme 2: Operational Flexibility Market Analysis
Apply the trained model to determine flexibility market needs.

---

**Contents:**
1. Introduction & Setup
2. Data Acquisition
3. Initial Visualisation
4. Quality Control
5. Statistical Analysis
6. Feature Engineering
7. Model Training
8. Probabilistic Forecasting
9. Flexibility Market Analysis
10. Summary & Conclusions

---
## 1. Introduction & Setup

We start by importing the required libraries and configuring our API access to the UK Power Networks Open Data Portal.

In [1]:
# Core imports
import os
import warnings
from datetime import datetime, timedelta
from types import SimpleNamespace

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch

# Machine learning
from sklearn.linear_model import LinearRegression, QuantileRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# UK Power Networks package
from ukpyn import ltds
from ukpyn.orchestrators import powerflow
from ukpyn.utils import (
    records_to_timeseries,
    detect_step_changes,
    identify_gaps,
    fill_gaps,
    flag_outliers,
    quality_control,
)

# Optional: holidays for feature engineering
try:
    import holidays
    UK_HOLIDAYS = holidays.UK()
    HAS_HOLIDAYS = True
except ImportError:
    HAS_HOLIDAYS = False
    print("Note: 'holidays' package not installed. Bank holiday features will be unavailable.")

warnings.filterwarnings('ignore')

print("All imports successful!")

All imports successful!


### Colour Palette

We use a consistent colour palette throughout this notebook for clear visual communication.

In [2]:
# Visualisation style guide colours
COLORS = {
    'primary': '#1f4e79',      # Navy blue - primary data
    'filled_gaps': '#2aa198',  # Teal - filled gaps
    'anomalies': '#dc322f',    # Coral - anomalies/removed
    'forecast': '#859900',     # Green - forecast
    'p90_band': '#859900',     # Light green with alpha for P90
    'rating': '#cb4b16',       # Orange - rating line
    'secondary': '#6c757d',    # Gray - secondary data
}

# Set matplotlib style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14

### API Key Configuration

Configure your UKPN Open Data Portal API key. You can either:
1. Set the `UKPN_API_KEY` environment variable
2. Create a `.env` file with `UKPN_API_KEY=your_key`
3. Run without a key to use synthetic fallback data for API-dependent sections

In [3]:
# Load API key from environment or .env file
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # python-dotenv not installed, will use environment variable directly

api_key = os.getenv('UKPN_API_KEY')
HAS_API_KEY = bool(api_key)

if HAS_API_KEY:
    print("API key loaded successfully.")
else:
    print("WARNING: No API key found. Set UKPN_API_KEY environment variable.")
    print("Notebook will continue using synthetic fallback data where needed.")

API key loaded successfully.


---
## 2. Data Acquisition

In this section, we:
1. Select a substation from LTDS Table 3A
2. Get transformer specifications from LTDS Table 2A/2B
3. Retrieve powerflow time series data

### 2.1 Select a Substation

We'll browse available primary substations from LTDS Table 3A and select one for analysis.

In [4]:
# Get available substations from LTDS Table 3A (observed peak demand)
# Filter to EPN licence area for this example
if HAS_API_KEY:
    try:
        table_3a_response = ltds.get_table_3a(licence_area='EPN', limit=100)
    except Exception as exc:
        print(f"Could not fetch LTDS Table 3A ({exc}). Falling back to synthetic path.")
        table_3a_response = SimpleNamespace(records=[], total_count=0)
else:
    table_3a_response = SimpleNamespace(records=[], total_count=0)
    print("Skipping LTDS Table 3A API call because UKPN_API_KEY is not set.")

print(f"Found {table_3a_response.total_count} substations in EPN licence area")
print(f"Retrieved {len(table_3a_response.records)} records")

# Convert to DataFrame for easier browsing
substations_df = pd.DataFrame([r.fields for r in table_3a_response.records])
substations_df.head(10)

Could not fetch LTDS Table 3A ([400] ODSQL query is malformed: Unknown field: licence_area. Clause(s) containing the error(s): where.). Falling back to synthetic path.
Found 0 substations in EPN licence area
Retrieved 0 records


""


In [5]:
# Select a substation for analysis
# Choose one with good data availability and moderate load

# For this example, we'll select the first available substation
# In practice, you might want to select based on specific criteria
if len(substations_df) > 0:
    # Try to find a substation with a recognizable name
    SELECTED_SUBSTATION = substations_df.iloc[0].get('substation', 'Unknown')
    print(f"Selected substation: {SELECTED_SUBSTATION}")
else:
    SELECTED_SUBSTATION = "Example Substation"
    print("No substations found in API response. Using placeholder.")

No substations found in API response. Using placeholder.


### 2.2 Get Transformer Data from LTDS Table 2

We retrieve transformer specifications to understand the substation's capacity and calculate N-1 ratings.

In [6]:
# Get 2-winding transformer data (Table 2A)
if HAS_API_KEY:
    try:
        table_2a_response = ltds.get_table_2a(
            substation=SELECTED_SUBSTATION,
            limit=50
)
    except Exception as exc:
        print(f"Could not fetch LTDS Table 2A ({exc}). Using empty fallback.")
        table_2a_response = SimpleNamespace(records=[], total_count=0)

    try:
        table_2b_response = ltds.get_table_2b(
            substation=SELECTED_SUBSTATION,
            limit=50
)
    except Exception as exc:
        print(f"Could not fetch LTDS Table 2B ({exc}). Using empty fallback.")
        table_2b_response = SimpleNamespace(records=[], total_count=0)
else:
    table_2a_response = SimpleNamespace(records=[], total_count=0)
    table_2b_response = SimpleNamespace(records=[], total_count=0)
    print("Skipping LTDS transformer API calls because UKPN_API_KEY is not set.")

print(f"Found {len(table_2a_response.records)} 2-winding transformers")
print(f"Found {len(table_2b_response.records)} 3-winding transformers")

Could not fetch LTDS Table 2A ([400] ODSQL query is malformed: Unknown field: substation. Clause(s) containing the error(s): where.). Using empty fallback.
Could not fetch LTDS Table 2B ([400] ODSQL query is malformed: Unknown field: substation. Clause(s) containing the error(s): where.). Using empty fallback.
Found 0 2-winding transformers
Found 0 3-winding transformers


In [7]:
# Extract transformer ratings and calculate N-1 capacity
transformer_ratings = []

# Process 2-winding transformers
for record in table_2a_response.records:
    fields = record.fields
    rating = fields.get('rating_mva') or fields.get('rated_mva') or fields.get('mva_rating')
    if rating:
        transformer_ratings.append({
            'id': fields.get('transformer_id', 'Unknown'),
            'type': '2-winding',
            'rating_mva': float(rating)
        })

# Process 3-winding transformers
for record in table_2b_response.records:
    fields = record.fields
    rating = fields.get('rating_mva') or fields.get('rated_mva') or fields.get('mva_rating')
    if rating:
        transformer_ratings.append({
            'id': fields.get('transformer_id', 'Unknown'),
            'type': '3-winding',
            'rating_mva': float(rating)
        })

transformers_df = pd.DataFrame(transformer_ratings)

if len(transformers_df) > 0:
    print("\nTransformer Summary:")
    print(transformers_df)
    
    # Calculate ratings
    total_rating = transformers_df['rating_mva'].sum()
    largest_tx = transformers_df['rating_mva'].max()
    n_minus_1_rating = total_rating - largest_tx
    
    print(f"\nTotal installed capacity: {total_rating:.1f} MVA")
    print(f"Largest transformer: {largest_tx:.1f} MVA")
    print(f"N-1 Rating: {n_minus_1_rating:.1f} MVA")
else:
    # Use placeholder values for demonstration
    print("No transformer data found. Using placeholder values.")
    total_rating = 60.0
    largest_tx = 30.0
    n_minus_1_rating = 30.0
    print(f"\nPlaceholder - Total: {total_rating} MVA, N-1: {n_minus_1_rating} MVA")

No transformer data found. Using placeholder values.

Placeholder - Total: 60.0 MVA, N-1: 30.0 MVA


### 2.3 Get Powerflow Time Series Data

We retrieve 2 years of half-hourly powerflow data for the selected substation's transformers.

In [8]:
# Define date range: 2 years of data (2023-2025)
START_DATE = '2023-01-01'
END_DATE = '2025-01-01'

print(f"Fetching powerflow data from {START_DATE} to {END_DATE}")

# Get primary transformer time series data
if HAS_API_KEY:
    try:
        powerflow_response = powerflow.get_transformer_timeseries(
            transformer_type='primary',
            granularity='half_hourly',
            licence_area='EPN',
            start_date=START_DATE,
            end_date=END_DATE,
            limit=50000  # Adjust based on expected data volume
)
    except Exception as exc:
        print(f"Could not fetch powerflow data ({exc}). Using synthetic fallback.")
        powerflow_response = SimpleNamespace(records=[], total_count=0)
else:
    powerflow_response = SimpleNamespace(records=[], total_count=0)
    print("Skipping powerflow API call because UKPN_API_KEY is not set.")

print(f"Retrieved {len(powerflow_response.records)} records")
print(f"Total available: {powerflow_response.total_count}")

Fetching powerflow data from 2023-01-01 to 2025-01-01
Could not fetch powerflow data (asyncio.run() cannot be called from a running event loop). Using synthetic fallback.
Retrieved 0 records
Total available: 0


In [9]:
# Convert to pandas DataFrame for analysis
if len(powerflow_response.records) > 0:
    # Use ukpyn utility to convert records
    powerflow_df = records_to_timeseries(powerflow_response.records)
    print(f"\nDataFrame shape: {powerflow_df.shape}")
    print(f"\nColumns: {list(powerflow_df.columns)}")
    print(f"\nDate range: {powerflow_df.index.min()} to {powerflow_df.index.max()}")
    powerflow_df.head()

In [10]:
# If no real data available, generate synthetic data for demonstration
if len(powerflow_response.records) == 0:
    print("Generating synthetic data for demonstration...")
    
    # Create date range (half-hourly)
    date_range = pd.date_range(start=START_DATE, end=END_DATE, freq='30min')
    
    # Generate realistic load profile
    np.random.seed(42)
    n_points = len(date_range)
    
    # Base load with daily and seasonal patterns
    hours = date_range.hour + date_range.minute / 60
    day_of_week = date_range.dayofweek
    day_of_year = date_range.dayofyear
    
    # Daily pattern (higher during day, lower at night)
    daily_pattern = 10 + 8 * np.sin((hours - 6) * np.pi / 12)
    daily_pattern = np.clip(daily_pattern, 5, 20)
    
    # Weekly pattern (lower on weekends)
    weekend_factor = np.where(day_of_week >= 5, 0.85, 1.0)
    
    # Seasonal pattern (higher in winter)
    seasonal_pattern = 1 + 0.3 * np.cos(2 * np.pi * (day_of_year - 15) / 365)
    
    # Combine patterns
    base_load = daily_pattern * weekend_factor * seasonal_pattern
    
    # Add noise and occasional spikes
    noise = np.random.normal(0, 1.5, n_points)
    spikes = np.random.choice([0, 5], size=n_points, p=[0.995, 0.005])
    
    # Simulate two transformers
    t1_load = (base_load + noise) * 0.55 + spikes * 0.3
    t2_load = (base_load + np.random.normal(0, 1.2, n_points)) * 0.45 + spikes * 0.7
    
    # Introduce some gaps
    gap_indices = np.random.choice(n_points, size=int(n_points * 0.002), replace=False)
    t1_load[gap_indices] = np.nan
    
    # Create DataFrame
    powerflow_df = pd.DataFrame({
        'active_power_t1_mw': t1_load,
        'active_power_t2_mw': t2_load,
        'transformer_id': 'T1'
    }, index=date_range)
    powerflow_df.index.name = 'timestamp'
    
    # Calculate total substation load
    powerflow_df['total_mw'] = powerflow_df['active_power_t1_mw'] + powerflow_df['active_power_t2_mw']
    
    print(f"Generated {len(powerflow_df)} synthetic data points")
    print(f"Date range: {powerflow_df.index.min()} to {powerflow_df.index.max()}")
    
    # Update ratings for synthetic data
    total_rating = 60.0
    n_minus_1_rating = 30.0

Generating synthetic data for demonstration...


TypeError: Index does not support mutable operations

In [ ]:
# Prepare the load time series for analysis
# Use total_mw if available, otherwise try to identify the power column

if 'total_mw' in powerflow_df.columns:
    load_series = powerflow_df['total_mw'].copy()
elif 'active_power_mw' in powerflow_df.columns:
    load_series = powerflow_df['active_power_mw'].copy()
else:
    # Try to find power columns and sum them
    power_cols = [c for c in powerflow_df.columns if 'mw' in c.lower() or 'power' in c.lower()]
    if power_cols:
        load_series = powerflow_df[power_cols].sum(axis=1)
    else:
        # Use first numeric column as fallback
        numeric_cols = powerflow_df.select_dtypes(include=[np.number]).columns
        load_series = powerflow_df[numeric_cols[0]].copy() if len(numeric_cols) > 0 else pd.Series(dtype=float)

load_series.name = 'load_mw'
print(f"\nLoad time series prepared:")
print(f"  Points: {len(load_series)}")
print(f"  Non-null: {load_series.notna().sum()}")
print(f"  Range: {load_series.min():.2f} to {load_series.max():.2f} MW")

---
## 3. Initial Visualisation

Create a stacked area chart showing the load across transformers over time.

In [ ]:
# Prepare data for stacked area chart
# Identify transformer columns
tx_cols = [c for c in powerflow_df.columns if 'active_power' in c.lower() and ('t1' in c.lower() or 't2' in c.lower() or 'tx' in c.lower())]

if len(tx_cols) == 0:
    tx_cols = ['active_power_t1_mw', 'active_power_t2_mw'] if 'active_power_t1_mw' in powerflow_df.columns else []

if len(tx_cols) >= 2:
    # Resample to daily for clearer visualization over 2 years
    daily_load = powerflow_df[tx_cols].resample('D').mean()
    
    fig, ax = plt.subplots(figsize=(16, 7))
    
    # Create stacked area chart
    colors = [COLORS['primary'], '#4a7ba7', '#7aa3c7']  # Gradient of blues
    ax.stackplot(daily_load.index, 
                 [daily_load[c].fillna(0) for c in tx_cols],
                 labels=[c.replace('active_power_', '').replace('_mw', '').upper() for c in tx_cols],
                 colors=colors[:len(tx_cols)],
                 alpha=0.8)
    
    # Add total line
    total_daily = daily_load.sum(axis=1)
    ax.plot(daily_load.index, total_daily, color='black', linewidth=1.5, label='Total', linestyle='--')
    
    # Add N-1 rating line
    ax.axhline(y=n_minus_1_rating, color=COLORS['rating'], linestyle='--', linewidth=2, label=f'N-1 Rating ({n_minus_1_rating:.0f} MW)')
    
    ax.set_xlabel('Date')
    ax.set_ylabel('Power (MW)')
    ax.set_title(f'Substation Load: {SELECTED_SUBSTATION}\n{START_DATE} to {END_DATE}')
    ax.legend(loc='upper right')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    # Simple time series plot
    daily_load = load_series.resample('D').mean()
    
    fig, ax = plt.subplots(figsize=(16, 7))
    ax.fill_between(daily_load.index, 0, daily_load, color=COLORS['primary'], alpha=0.7, label='Total Load')
    ax.axhline(y=n_minus_1_rating, color=COLORS['rating'], linestyle='--', linewidth=2, label=f'N-1 Rating ({n_minus_1_rating:.0f} MW)')
    
    ax.set_xlabel('Date')
    ax.set_ylabel('Power (MW)')
    ax.set_title(f'Substation Load: {SELECTED_SUBSTATION}\n{START_DATE} to {END_DATE}')
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

---
## 4. Quality Control

Apply quality control procedures using `ukpyn.utils.timeseries`:
1. Step change detection
2. Gap identification and filling
3. Anomaly detection and removal

### 4.1 Step Change Detection

Identify sudden shifts in the data that may indicate equipment changes or abnormal running arrangements.

In [ ]:
# Detect step changes in the load time series
step_changes = detect_step_changes(
    load_series,
    threshold=0.15,      # 15% relative change threshold
    window_size=48,      # 24-hour window (48 half-hourly points)
    min_confidence=0.75
)

print(f"Detected {len(step_changes)} step changes:\n")
for change in step_changes[:5]:  # Show first 5
    print(f"  {change.timestamp.strftime('%Y-%m-%d %H:%M')}: "
          f"{change.direction} of {change.relative_change:.1%} "
          f"({change.value_before:.1f} -> {change.value_after:.1f} MW) "
          f"[confidence: {change.confidence:.2f}]")

In [ ]:
# Visualize step changes
if len(step_changes) > 0:
    fig, ax = plt.subplots(figsize=(16, 7))
    
    # Plot daily average for clarity
    daily_load = load_series.resample('D').mean()
    ax.plot(daily_load.index, daily_load.values, color=COLORS['primary'], linewidth=1, label='Daily Average Load')
    
    # Highlight step change regions
    for change in step_changes:
        # Add vertical band around step change
        start = change.timestamp - timedelta(days=2)
        end = change.timestamp + timedelta(days=2)
        ax.axvspan(start, end, alpha=0.3, 
                   color=COLORS['anomalies'] if change.direction == 'decrease' else COLORS['filled_gaps'],
                   label=f'Step Change ({change.direction})' if change == step_changes[0] else '')
        
        # Add annotation
        ax.annotate(f"{change.relative_change:.0%}",
                    xy=(change.timestamp, change.value_after),
                    xytext=(change.timestamp, change.value_after + 3),
                    fontsize=9, ha='center')
    
    ax.set_xlabel('Date')
    ax.set_ylabel('Power (MW)')
    ax.set_title(f'Step Change Detection - {SELECTED_SUBSTATION}')
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()
else:
    print("No significant step changes detected in the data.")

### 4.2 Gap Identification and Filling

Identify data gaps and fill them using appropriate interpolation methods.

In [ ]:
# Identify gaps in the time series
gaps = identify_gaps(
    load_series,
    expected_frequency='30min',
    min_gap_hours=1.0
)

print(f"Identified {len(gaps)} data gaps:\n")
total_gap_hours = sum(g.duration_hours for g in gaps)
print(f"Total gap duration: {total_gap_hours:.1f} hours")

if len(gaps) > 0:
    print("\nLargest gaps:")
    sorted_gaps = sorted(gaps, key=lambda x: x.duration_hours, reverse=True)
    for gap in sorted_gaps[:5]:
        print(f"  {gap.start.strftime('%Y-%m-%d %H:%M')} to {gap.end.strftime('%Y-%m-%d %H:%M')}: "
              f"{gap.duration_hours:.1f} hours ({gap.missing_points} points)")

In [ ]:
# Fill gaps using linear interpolation for gaps up to 48 hours (2 days)
load_filled = fill_gaps(
    load_series,
    method='linear',
    max_gap_hours=48.0
)

# Track which points were filled
filled_mask = load_series.isna() & load_filled.notna()

print(f"\nGap filling results:")
print(f"  Original missing: {load_series.isna().sum()}")
print(f"  Remaining missing: {load_filled.isna().sum()}")
print(f"  Points filled: {filled_mask.sum()}")

In [ ]:
# Visualize gap filling - zoom to a window with filled gaps
if filled_mask.sum() > 0:
    # Find a window with filled gaps
    filled_indices = filled_mask[filled_mask].index
    gap_center = filled_indices[len(filled_indices)//2]
    window_start = gap_center - timedelta(days=2.5)
    window_end = gap_center + timedelta(days=2.5)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Window data
    window_original = load_series[window_start:window_end]
    window_filled = load_filled[window_start:window_end]
    window_mask = filled_mask[window_start:window_end]
    
    # Plot original data
    ax.plot(window_original.index, window_original.values, 
            color=COLORS['primary'], linewidth=1.5, label='Original Data', marker='o', markersize=2)
    
    # Plot filled data (where it differs)
    filled_points = window_filled[window_mask]
    ax.plot(filled_points.index, filled_points.values, 
            color=COLORS['filled_gaps'], linewidth=2, linestyle='--', 
            label='Filled Data', marker='s', markersize=4)
    
    ax.set_xlabel('Date')
    ax.set_ylabel('Power (MW)')
    ax.set_title(f'Gap Filling Example - 5 Day Window\n{window_start.strftime("%Y-%m-%d")} to {window_end.strftime("%Y-%m-%d")}')
    ax.legend(loc='upper right')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %H:%M'))
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No gaps were filled (no gaps found or all gaps exceeded max_gap_hours).")

### 4.3 Anomaly Detection and Removal

Flag outliers using IQR method and create a clean dataset for forecasting.

In [ ]:
# Flag outliers using IQR method
outlier_flags = flag_outliers(
    load_filled,
    method='iqr',
    threshold=2.5
)

n_outliers = outlier_flags.sum()
print(f"Detected {n_outliers} outliers ({100*n_outliers/len(outlier_flags):.2f}%)")

if n_outliers > 0:
    outlier_values = load_filled[outlier_flags]
    print(f"\nOutlier statistics:")
    print(f"  Min outlier: {outlier_values.min():.2f} MW")
    print(f"  Max outlier: {outlier_values.max():.2f} MW")

In [ ]:
# Run comprehensive quality control
qc_report = quality_control(
    load_filled,
    expected_frequency='30min',
    outlier_method='iqr',
    outlier_threshold=2.5
)

print("Quality Control Report")
print("=" * 40)
print(f"Total points: {qc_report.total_points:,}")
print(f"Valid points: {qc_report.valid_points:,}")
print(f"Missing points: {qc_report.missing_points}")
print(f"Outlier points: {qc_report.outlier_points}")
print(f"Data gaps: {len(qc_report.gaps)}")
print(f"\nQuality Score: {qc_report.quality_score:.1f}%")

if qc_report.issues:
    print("\nIssues identified:")
    for issue in qc_report.issues:
        print(f"  - {issue}")

In [ ]:
# Visualize anomaly detection: clean data vs removed points
fig, ax = plt.subplots(figsize=(16, 7))

# Sample data for visualization (monthly sample for clarity)
sample_period = load_filled['2024-01':'2024-01']  # January 2024
sample_outliers = outlier_flags['2024-01':'2024-01']

# Plot clean data
clean_data = sample_period[~sample_outliers]
ax.plot(clean_data.index, clean_data.values, 
        color=COLORS['secondary'], linewidth=0.8, alpha=0.8, label='Clean Data')

# Plot outliers
outlier_data = sample_period[sample_outliers]
ax.scatter(outlier_data.index, outlier_data.values, 
           color=COLORS['anomalies'], s=30, zorder=5, label='Anomalies (Removed)', marker='x')

ax.set_xlabel('Date')
ax.set_ylabel('Power (MW)')
ax.set_title('Anomaly Detection - January 2024')
ax.legend(loc='upper right')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
plt.tight_layout()
plt.show()

In [ ]:
# Create clean time series for forecasting
# Replace outliers with NaN and then interpolate
load_clean = load_filled.copy()
load_clean[outlier_flags] = np.nan
load_clean = fill_gaps(load_clean, method='linear', max_gap_hours=4.0)

print(f"Clean time series prepared:")
print(f"  Original points: {len(load_series)}")
print(f"  Clean points: {load_clean.notna().sum()}")
print(f"  Remaining NaN: {load_clean.isna().sum()}")

---
## 5. Statistical Analysis

Perform statistical analysis to understand the load characteristics.

In [ ]:
# Helper functions for statistical analysis
# (These would normally be in ukpyn.utils.stats module)

def describe_timeseries(series):
    """Comprehensive statistical summary."""
    from scipy import stats as scipy_stats
    
    clean = series.dropna()
    
    return {
        'count': len(clean),
        'mean': clean.mean(),
        'median': clean.median(),
        'std': clean.std(),
        'min': clean.min(),
        'max': clean.max(),
        'range': clean.max() - clean.min(),
        'skewness': scipy_stats.skew(clean),
        'kurtosis': scipy_stats.kurtosis(clean),
        'P10': clean.quantile(0.10),
        'P25': clean.quantile(0.25),
        'P50': clean.quantile(0.50),
        'P75': clean.quantile(0.75),
        'P90': clean.quantile(0.90),
        'P95': clean.quantile(0.95),
        'P99': clean.quantile(0.99),
    }

def autocorrelation(series, lags=48):
    """Calculate autocorrelation for given lags."""
    clean = series.dropna()
    acf = [clean.autocorr(lag=i) for i in range(lags + 1)]
    return pd.Series(acf, index=range(lags + 1))

def seasonal_pattern(series, period='daily'):
    """Extract seasonal patterns."""
    clean = series.dropna()
    
    if period == 'daily':
        return clean.groupby(clean.index.hour).agg(['mean', 'std', 'min', 'max'])
    elif period == 'weekly':
        return clean.groupby(clean.index.dayofweek).agg(['mean', 'std', 'min', 'max'])
    elif period == 'annual':
        return clean.groupby(clean.index.month).agg(['mean', 'std', 'min', 'max'])
    else:
        raise ValueError(f"Unknown period: {period}")

def peak_analysis(series, threshold_percentile=90):
    """Analyze peak characteristics."""
    clean = series.dropna()
    threshold = clean.quantile(threshold_percentile / 100)
    
    peaks = clean[clean > threshold]
    
    return {
        'threshold': threshold,
        'threshold_percentile': threshold_percentile,
        'peak_count': len(peaks),
        'peak_mean': peaks.mean(),
        'peak_max': peaks.max(),
        'peak_hours': peaks.groupby(peaks.index.hour).size().to_dict(),
        'peak_days': peaks.groupby(peaks.index.dayofweek).size().to_dict(),
    }

In [ ]:
# Run comprehensive statistical analysis
stats = describe_timeseries(load_clean)

print("Statistical Summary")
print("=" * 40)
print(f"Count:      {stats['count']:,}")
print(f"Mean:       {stats['mean']:.2f} MW")
print(f"Median:     {stats['median']:.2f} MW")
print(f"Std Dev:    {stats['std']:.2f} MW")
print(f"Min:        {stats['min']:.2f} MW")
print(f"Max:        {stats['max']:.2f} MW")
print(f"Range:      {stats['range']:.2f} MW")
print(f"\nDistribution:")
print(f"  Skewness: {stats['skewness']:.3f}")
print(f"  Kurtosis: {stats['kurtosis']:.3f}")
print(f"\nPercentiles:")
print(f"  P10:  {stats['P10']:.2f} MW")
print(f"  P25:  {stats['P25']:.2f} MW")
print(f"  P50:  {stats['P50']:.2f} MW")
print(f"  P75:  {stats['P75']:.2f} MW")
print(f"  P90:  {stats['P90']:.2f} MW")
print(f"  P95:  {stats['P95']:.2f} MW")

In [ ]:
# Autocorrelation analysis
acf = autocorrelation(load_clean, lags=96)  # 48 hours at half-hourly resolution

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(acf.index, acf.values, color=COLORS['primary'], alpha=0.7)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.axhline(y=1.96/np.sqrt(len(load_clean)), color='red', linestyle='--', linewidth=1, label='95% CI')
ax.axhline(y=-1.96/np.sqrt(len(load_clean)), color='red', linestyle='--', linewidth=1)

# Mark key lags
ax.axvline(x=48, color=COLORS['rating'], linestyle=':', alpha=0.7, label='24h (lag 48)')

ax.set_xlabel('Lag (half-hours)')
ax.set_ylabel('Autocorrelation')
ax.set_title('Autocorrelation Function (ACF)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nKey autocorrelations:")
print(f"  Lag 1 (30 min):  {acf[1]:.3f}")
print(f"  Lag 2 (1 hour):  {acf[2]:.3f}")
print(f"  Lag 48 (24 hr):  {acf[48]:.3f}")

In [ ]:
# Seasonal patterns
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Daily pattern
daily = seasonal_pattern(load_clean, 'daily')
ax = axes[0]
ax.fill_between(daily.index, daily['min'], daily['max'], alpha=0.2, color=COLORS['primary'])
ax.plot(daily.index, daily['mean'], color=COLORS['primary'], linewidth=2, label='Mean')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Power (MW)')
ax.set_title('Daily Profile')
ax.set_xticks(range(0, 24, 3))

# Weekly pattern
weekly = seasonal_pattern(load_clean, 'weekly')
ax = axes[1]
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
ax.bar(range(7), weekly['mean'], color=COLORS['primary'], alpha=0.7, yerr=weekly['std'], capsize=3)
ax.set_xlabel('Day of Week')
ax.set_ylabel('Power (MW)')
ax.set_title('Weekly Profile')
ax.set_xticks(range(7))
ax.set_xticklabels(days)

# Annual pattern
annual = seasonal_pattern(load_clean, 'annual')
ax = axes[2]
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
ax.fill_between(annual.index, annual['min'], annual['max'], alpha=0.2, color=COLORS['primary'])
ax.plot(annual.index, annual['mean'], color=COLORS['primary'], linewidth=2, marker='o')
ax.set_xlabel('Month')
ax.set_ylabel('Power (MW)')
ax.set_title('Annual Profile')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(months, rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Peak analysis
peaks = peak_analysis(load_clean, threshold_percentile=90)

print("Peak Load Analysis (>P90)")
print("=" * 40)
print(f"Threshold: {peaks['threshold']:.2f} MW (P{peaks['threshold_percentile']})")
print(f"Peak events: {peaks['peak_count']}")
print(f"Peak mean: {peaks['peak_mean']:.2f} MW")
print(f"Peak max: {peaks['peak_max']:.2f} MW")

print(f"\nPeak hours distribution:")
for hour, count in sorted(peaks['peak_hours'].items()):
    print(f"  {hour:02d}:00 - {count} occurrences")

---
## 6. Feature Engineering

Create features for machine learning models:
1. Temporal features
2. Weather features (optional)
3. Lagged features

In [ ]:
# Create feature DataFrame
df = pd.DataFrame({'load_mw': load_clean})
df = df.dropna()  # Remove any remaining NaN values

print(f"Preparing features for {len(df)} data points")

### 6.1 Temporal Features

In [ ]:
# Temporal features
df['hour'] = df.index.hour
df['minute'] = df.index.minute
df['day_of_week'] = df.index.dayofweek
df['day_of_month'] = df.index.day
df['month'] = df.index.month
df['quarter'] = df.index.quarter
df['year'] = df.index.year
df['is_weekend'] = (df.index.dayofweek >= 5).astype(int)

# Cyclical encoding for hour and day of week
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# UK Bank Holidays
if HAS_HOLIDAYS:
    df['is_holiday'] = df.index.map(lambda x: x.date() in UK_HOLIDAYS).astype(int)
else:
    df['is_holiday'] = 0

print(f"Temporal features created: {df.shape[1] - 1} features")

### 6.2 Weather Features (Optional)

Weather data could be added from external sources. For now, we'll skip this or use placeholders.

In [ ]:
# Weather features would be added here if available
# Example structure:
# df['temperature_c'] = weather_data['temperature']
# df['humidity_pct'] = weather_data['humidity']
# df['wind_speed_ms'] = weather_data['wind_speed']
# df['solar_irradiance_wm2'] = weather_data['ghi']

print("Weather features: Not available (optional integration)")
print("To add weather data, use ukpyn.integrations.weather (if available) or external APIs.")

### 6.3 Lagged Features

In [ ]:
# Lagged features
# t-1 (30 min ago)
df['load_lag_1'] = df['load_mw'].shift(1)
# t-2 (1 hour ago)
df['load_lag_2'] = df['load_mw'].shift(2)
# t-4 (2 hours ago)
df['load_lag_4'] = df['load_mw'].shift(4)
# t-48 (24 hours ago - same time yesterday)
df['load_lag_48'] = df['load_mw'].shift(48)
# t-336 (same time last week)
df['load_lag_336'] = df['load_mw'].shift(336)

# Rolling statistics
df['load_rolling_mean_4h'] = df['load_mw'].rolling(window=8).mean()
df['load_rolling_std_4h'] = df['load_mw'].rolling(window=8).std()
df['load_rolling_mean_24h'] = df['load_mw'].rolling(window=48).mean()

# Difference features
df['load_diff_1'] = df['load_mw'].diff(1)
df['load_diff_48'] = df['load_mw'].diff(48)

print(f"Lagged features created. Total features: {df.shape[1] - 1}")

In [ ]:
# Drop rows with NaN values from lagging
df_features = df.dropna()

print(f"Final dataset: {len(df_features)} rows, {df_features.shape[1]} columns")
print(f"\nFeature columns:")
feature_cols = [c for c in df_features.columns if c != 'load_mw']
print(feature_cols)

---
## 7. Model Training

Train multiple forecasting models and compare their performance.

### 7.1 Data Preparation

Split data into training and testing sets, ensuring we split on whole days to avoid data leakage.

In [ ]:
# Use first year for training, second year for testing
# This simulates a realistic forecasting scenario

train_end = '2024-01-01'
test_start = '2024-01-01'

train_df = df_features[df_features.index < train_end]
test_df = df_features[df_features.index >= test_start]

# Prepare X and y
X_train = train_df[feature_cols]
y_train = train_df['load_mw']

X_test = test_df[feature_cols]
y_test = test_df['load_mw']

print(f"Training set: {len(train_df)} samples ({train_df.index.min().date()} to {train_df.index.max().date()})")
print(f"Testing set: {len(test_df)} samples ({test_df.index.min().date()} to {test_df.index.max().date()})")
print(f"\nTrain/Test split: {100*len(train_df)/(len(train_df)+len(test_df)):.1f}% / {100*len(test_df)/(len(train_df)+len(test_df)):.1f}%")

In [ ]:
# Scale features for models that benefit from scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 7.2 Model Training

In [ ]:
# Dictionary to store models and results
models = {}
predictions = {}
metrics = {}

def evaluate_model(name, y_true, y_pred):
    """Calculate evaluation metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = 100 * np.mean(np.abs((y_true - y_pred) / y_true))
    r2 = r2_score(y_true, y_pred)
    
    return {
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape,
        'R2': r2
    }

In [ ]:
# Model 1: Smart Persistence (same time yesterday)
print("Training Model 1: Smart Persistence...")

# Use lag_48 as prediction (same time yesterday)
y_pred_persistence = X_test['load_lag_48'].values
predictions['Smart Persistence'] = y_pred_persistence
metrics['Smart Persistence'] = evaluate_model('Smart Persistence', y_test, y_pred_persistence)

print(f"  RMSE: {metrics['Smart Persistence']['RMSE']:.3f} MW")

In [ ]:
# Model 2: Linear Regression
print("Training Model 2: Linear Regression...")

lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
models['Linear Regression'] = lr_model

y_pred_lr = lr_model.predict(X_test_scaled)
predictions['Linear Regression'] = y_pred_lr
metrics['Linear Regression'] = evaluate_model('Linear Regression', y_test, y_pred_lr)

print(f"  RMSE: {metrics['Linear Regression']['RMSE']:.3f} MW")

In [ ]:
# Model 3: Random Forest
print("Training Model 3: Random Forest...")

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    n_jobs=-1,
    random_state=42
)
rf_model.fit(X_train, y_train)  # RF doesn't need scaling
models['Random Forest'] = rf_model

y_pred_rf = rf_model.predict(X_test)
predictions['Random Forest'] = y_pred_rf
metrics['Random Forest'] = evaluate_model('Random Forest', y_test, y_pred_rf)

print(f"  RMSE: {metrics['Random Forest']['RMSE']:.3f} MW")

In [ ]:
# Model 4: Gradient Boosting
print("Training Model 4: Gradient Boosting...")

gb_model = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    min_samples_split=10,
    random_state=42
)
gb_model.fit(X_train, y_train)
models['Gradient Boosting'] = gb_model

y_pred_gb = gb_model.predict(X_test)
predictions['Gradient Boosting'] = y_pred_gb
metrics['Gradient Boosting'] = evaluate_model('Gradient Boosting', y_test, y_pred_gb)

print(f"  RMSE: {metrics['Gradient Boosting']['RMSE']:.3f} MW")

### 7.3 Model Comparison

In [ ]:
# Compare model performance
metrics_df = pd.DataFrame(metrics).T
metrics_df = metrics_df.round(3)

print("\nModel Performance Comparison")
print("=" * 60)
print(metrics_df.to_string())

# Identify best model
best_model_name = metrics_df['RMSE'].idxmin()
print(f"\nBest model (lowest RMSE): {best_model_name}")
print(f"  RMSE: {metrics_df.loc[best_model_name, 'RMSE']:.3f} MW")
print(f"  R2: {metrics_df.loc[best_model_name, 'R2']:.3f}")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of RMSE
ax = axes[0]
colors = [COLORS['primary'] if name != best_model_name else COLORS['forecast'] for name in metrics_df.index]
ax.barh(metrics_df.index, metrics_df['RMSE'], color=colors)
ax.set_xlabel('RMSE (MW)')
ax.set_title('Model Comparison - RMSE (lower is better)')
ax.invert_yaxis()

# Bar chart of R2
ax = axes[1]
ax.barh(metrics_df.index, metrics_df['R2'], color=colors)
ax.set_xlabel('R2 Score')
ax.set_title('Model Comparison - R2 (higher is better)')
ax.invert_yaxis()
ax.set_xlim([0, 1])

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (for tree-based models)
if 'Random Forest' in models:
    importance_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': models['Random Forest'].feature_importances_
    }).sort_values('importance', ascending=False)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    top_features = importance_df.head(15)
    ax.barh(top_features['feature'], top_features['importance'], color=COLORS['primary'])
    ax.set_xlabel('Feature Importance')
    ax.set_title('Top 15 Feature Importances (Random Forest)')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

---
## 8. Probabilistic Forecasting

Generate P10, P50, and P90 quantile predictions for uncertainty quantification.

In [ ]:
# Train quantile regressors
print("Training Quantile Regressors...")

quantiles = [0.10, 0.50, 0.90]
quantile_models = {}
quantile_predictions = {}

# Use a subset for faster training (quantile regression can be slow)
sample_size = min(10000, len(X_train_scaled))
sample_idx = np.random.choice(len(X_train_scaled), sample_size, replace=False)
X_train_sample = X_train_scaled[sample_idx]
y_train_sample = y_train.iloc[sample_idx]

for q in quantiles:
    print(f"  Training P{int(q*100)} model...")
    qr = QuantileRegressor(quantile=q, alpha=0.1, solver='highs')
    qr.fit(X_train_sample, y_train_sample)
    quantile_models[q] = qr
    quantile_predictions[f'P{int(q*100)}'] = qr.predict(X_test_scaled)

print("Quantile regression complete.")

In [ ]:
# Visualize probabilistic forecast for a sample period
# Select a week in the test period
sample_start = '2024-06-01'
sample_end = '2024-06-08'

sample_mask = (test_df.index >= sample_start) & (test_df.index < sample_end)
sample_idx_range = np.where(sample_mask)[0]

fig, ax = plt.subplots(figsize=(16, 7))

# Plot actual
sample_dates = test_df.index[sample_mask]
ax.plot(sample_dates, y_test.iloc[sample_idx_range], 
        color=COLORS['primary'], linewidth=1.5, label='Actual')

# Plot P50 forecast
ax.plot(sample_dates, quantile_predictions['P50'][sample_idx_range], 
        color=COLORS['forecast'], linewidth=1.5, linestyle='--', label='P50 Forecast')

# Plot P10-P90 band
ax.fill_between(sample_dates, 
                quantile_predictions['P10'][sample_idx_range],
                quantile_predictions['P90'][sample_idx_range],
                color=COLORS['p90_band'], alpha=0.3, label='P10-P90 Range')

ax.set_xlabel('Date')
ax.set_ylabel('Power (MW)')
ax.set_title(f'Probabilistic Forecast - {sample_start} to {sample_end}')
ax.legend(loc='upper right')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %H:%M'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate coverage of prediction intervals
p10 = quantile_predictions['P10']
p90 = quantile_predictions['P90']

coverage = np.mean((y_test.values >= p10) & (y_test.values <= p90))
print(f"P10-P90 Interval Coverage: {100*coverage:.1f}% (target: 80%)")

---
## 9. Flexibility Market Analysis

**Theme 2: Operational Flexibility Market Analysis**

Apply the trained model to identify days requiring flexibility procurement.

### 9.1 N-1 Rating Calculation

In [ ]:
# Recap N-1 rating from earlier
print("Transformer Capacity Summary")
print("=" * 40)
print(f"Total installed capacity: {total_rating:.1f} MVA")
print(f"Largest transformer: {largest_tx:.1f} MVA")
print(f"N-1 Rating (constraint): {n_minus_1_rating:.1f} MVA")
print(f"\nNote: N-1 rating represents the capacity available when the largest")
print(f"transformer is out of service. Load exceeding this threshold requires")
print(f"flexibility services to maintain security of supply.")

### 9.2 Daily 10am Forecast Scenario

Simulate an operational scenario where forecasts are made at 10am each day for the rest of the day.

In [ ]:
# Generate daily peak forecasts with P90
# Group by date and get daily maximum actual and forecast

daily_analysis = pd.DataFrame({
    'actual_peak': y_test.groupby(y_test.index.date).max(),
    'forecast_peak': pd.Series(predictions[best_model_name], index=y_test.index).groupby(y_test.index.date).max(),
    'p90_peak': pd.Series(quantile_predictions['P90'], index=y_test.index).groupby(y_test.index.date).max()
})

daily_analysis.index = pd.to_datetime(daily_analysis.index)

print(f"Daily analysis prepared for {len(daily_analysis)} days")
daily_analysis.head(10)

### 9.3 Identify Flex Market Need Days

Days where P90 forecast exceeds the N-1 rating are flagged for flexibility procurement.

In [ ]:
# Identify days where P90 forecast exceeds N-1 rating
daily_analysis['needs_flex'] = daily_analysis['p90_peak'] > n_minus_1_rating
daily_analysis['actual_exceeds'] = daily_analysis['actual_peak'] > n_minus_1_rating
daily_analysis['margin_mw'] = n_minus_1_rating - daily_analysis['p90_peak']

flex_days = daily_analysis[daily_analysis['needs_flex']]

print(f"Flexibility Market Analysis")
print("=" * 50)
print(f"Analysis period: {daily_analysis.index.min().date()} to {daily_analysis.index.max().date()}")
print(f"Total days analyzed: {len(daily_analysis)}")
print(f"\nDays requiring flexibility (P90 > N-1): {len(flex_days)}")
print(f"Days where actual exceeded N-1: {daily_analysis['actual_exceeds'].sum()}")

if len(flex_days) > 0:
    print(f"\nFlex need days breakdown:")
    print(f"  Average shortfall: {-flex_days['margin_mw'].mean():.2f} MW")
    print(f"  Maximum shortfall: {-flex_days['margin_mw'].min():.2f} MW")

In [ ]:
# List the flex-need days
if len(flex_days) > 0:
    print("\nFlexibility Market Need Days:")
    print("-" * 70)
    print(f"{'Date':<12} {'Actual Peak':>12} {'P90 Forecast':>14} {'N-1 Rating':>12} {'Shortfall':>12}")
    print("-" * 70)
    
    for idx, row in flex_days.head(20).iterrows():
        shortfall = row['p90_peak'] - n_minus_1_rating
        print(f"{idx.strftime('%Y-%m-%d'):<12} {row['actual_peak']:>10.2f} MW {row['p90_peak']:>12.2f} MW "
              f"{n_minus_1_rating:>10.1f} MW {shortfall:>10.2f} MW")
    
    if len(flex_days) > 20:
        print(f"... and {len(flex_days) - 20} more days")

### 9.4 Visualisation of Flex-Need Days

In [ ]:
# Time series of daily peaks with N-1 rating
fig, ax = plt.subplots(figsize=(16, 7))

# Plot actual peaks
ax.plot(daily_analysis.index, daily_analysis['actual_peak'], 
        color=COLORS['primary'], linewidth=1, alpha=0.7, label='Actual Daily Peak')

# Plot P90 forecast peaks
ax.plot(daily_analysis.index, daily_analysis['p90_peak'], 
        color=COLORS['forecast'], linewidth=1, alpha=0.7, label='P90 Daily Peak Forecast')

# N-1 rating line
ax.axhline(y=n_minus_1_rating, color=COLORS['rating'], linestyle='--', linewidth=2, 
           label=f'N-1 Rating ({n_minus_1_rating:.0f} MW)')

# Highlight flex-need days
if len(flex_days) > 0:
    # Shade regions where P90 exceeds rating
    ax.fill_between(daily_analysis.index, 
                    n_minus_1_rating,
                    daily_analysis['p90_peak'],
                    where=daily_analysis['p90_peak'] > n_minus_1_rating,
                    color=COLORS['anomalies'], alpha=0.3, 
                    label='Flexibility Requirement')

ax.set_xlabel('Date')
ax.set_ylabel('Peak Power (MW)')
ax.set_title(f'Flexibility Market Analysis - {SELECTED_SUBSTATION}')
ax.legend(loc='upper right')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed view of a specific flex-need day (if any exist)
if len(flex_days) > 0:
    # Select the day with highest shortfall
    worst_day = flex_days['margin_mw'].idxmin()
    
    # Get half-hourly data for that day
    day_mask = y_test.index.date == worst_day.date()
    day_idx = np.where(day_mask)[0]
    
    fig, ax = plt.subplots(figsize=(14, 7))
    
    day_times = y_test.index[day_mask]
    
    # Plot actual load
    ax.plot(day_times, y_test.iloc[day_idx], 
            color=COLORS['primary'], linewidth=2, label='Actual Load')
    
    # Plot point forecast
    ax.plot(day_times, predictions[best_model_name][day_idx], 
            color=COLORS['forecast'], linewidth=2, linestyle='--', label='Point Forecast')
    
    # Plot P90 forecast
    ax.plot(day_times, quantile_predictions['P90'][day_idx], 
            color=COLORS['forecast'], linewidth=1.5, linestyle=':', label='P90 Forecast')
    
    # N-1 rating line
    ax.axhline(y=n_minus_1_rating, color=COLORS['rating'], linestyle='--', linewidth=2, 
               label=f'N-1 Rating ({n_minus_1_rating:.0f} MW)')
    
    # Shade constraint violation period
    p90_vals = quantile_predictions['P90'][day_idx]
    ax.fill_between(day_times, n_minus_1_rating, p90_vals,
                    where=p90_vals > n_minus_1_rating,
                    color=COLORS['anomalies'], alpha=0.3,
                    label='Flex Requirement Period')
    
    ax.set_xlabel('Time')
    ax.set_ylabel('Power (MW)')
    ax.set_title(f'Flex Market Need Day: {worst_day.strftime("%Y-%m-%d")}\n'
                 f'Shortfall: {-flex_days.loc[worst_day, "margin_mw"]:.1f} MW')
    ax.legend(loc='upper left')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    plt.tight_layout()
    plt.show()
else:
    print("No flex-need days identified in the test period.")
    print("This could mean:")
    print("  - The substation is well within capacity limits")
    print("  - The N-1 rating may need adjustment for your analysis")

In [ ]:
# Monthly summary of flexibility requirements
monthly_flex = daily_analysis.groupby(daily_analysis.index.to_period('M')).agg({
    'needs_flex': 'sum',
    'actual_exceeds': 'sum',
    'margin_mw': lambda x: (x[x < 0]).sum() if (x < 0).any() else 0  # Total MW shortfall
})

monthly_flex.columns = ['Days Needing Flex', 'Days Actually Exceeding', 'Total Shortfall (MW)']
monthly_flex['Total Shortfall (MW)'] = -monthly_flex['Total Shortfall (MW)']  # Make positive

print("\nMonthly Flexibility Requirement Summary")
print("=" * 60)
print(monthly_flex.to_string())

---
## 10. Summary & Conclusions

This notebook demonstrated a comprehensive workflow for substation load forecasting using UK Power Networks open data.

In [ ]:
# Summary statistics
print("="*70)
print("SUBSTATION LOAD FORECASTING - SUMMARY")
print("="*70)

print(f"\n1. DATA")
print(f"   Substation: {SELECTED_SUBSTATION}")
print(f"   Period: {START_DATE} to {END_DATE}")
print(f"   Data points: {len(load_clean):,}")
print(f"   Quality score: {qc_report.quality_score:.1f}%")

print(f"\n2. CAPACITY")
print(f"   Total rating: {total_rating:.1f} MVA")
print(f"   N-1 rating: {n_minus_1_rating:.1f} MVA")

print(f"\n3. LOAD STATISTICS")
print(f"   Mean: {stats['mean']:.2f} MW")
print(f"   Peak: {stats['max']:.2f} MW")
print(f"   P90: {stats['P90']:.2f} MW")

print(f"\n4. FORECASTING PERFORMANCE")
print(f"   Best model: {best_model_name}")
print(f"   RMSE: {metrics[best_model_name]['RMSE']:.3f} MW")
print(f"   R2: {metrics[best_model_name]['R2']:.3f}")
print(f"   MAPE: {metrics[best_model_name]['MAPE']:.2f}%")

print(f"\n5. FLEXIBILITY MARKET ANALYSIS")
print(f"   Days analyzed: {len(daily_analysis)}")
print(f"   Days requiring flex: {len(flex_days)}")
print(f"   Percentage: {100*len(flex_days)/len(daily_analysis):.1f}%")

### Key Findings

**Theme 1: Raw Time Series Forecaster**
- Successfully built an end-to-end forecasting pipeline from data acquisition to model deployment
- Quality control procedures identified and addressed data gaps and anomalies
- Statistical analysis revealed clear daily, weekly, and seasonal load patterns
- Multiple ML models were trained and compared, with tree-based models typically performing best

**Theme 2: Operational Flexibility Market Analysis**
- N-1 rating provides a clear threshold for flexibility market needs
- Probabilistic forecasting (P90) enables conservative capacity planning
- Identified specific days requiring flexibility procurement
- Monthly summaries support operational planning and market participation

### Next Steps
1. **Integrate weather data** - Temperature and solar irradiance can significantly improve forecast accuracy
2. **Add more substations** - Apply this methodology across the network
3. **Refine P90 calibration** - Ensure 90% coverage in backtesting
4. **Operational deployment** - Automate daily forecast generation
5. **Flexibility market integration** - Connect forecasts to procurement systems

In [ ]:
print("\nNotebook completed successfully.")
print("For more information, visit: https://ukpowernetworks.opendatasoft.com")